# Identificación y autoría

**Speaker:** Eduardo Osorio Venegas  
**Apoyo de inteligencia artificial generativa:** ChatGPT (OpenAI).

**Nota de elaboración:** Las celdas Markdown cuentan con apoyo de inteligencia artificial generativa. Los ejemplos tienen fines educativos y demostrativos. El autor es responsable de su revisión, adaptación y validación en Microsoft Fabric.

# NB_04 — Generación de resúmenes con `ai.summarize`
**Serie:** Microsoft Fabric AI Functions con PySpark  
**Modelo:** `gpt-5-mini`

## Objetivos
- Resumir una columna de texto y comparar con un resumen ejecutivo basado en múltiples columnas.
- Validar errores, longitud y fidelidad de los resúmenes.
- Auditar tokens y CU-segundos por ejecución.

**Referencias oficiales:** [Summarize](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/summarize) · [Configuración](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/configuration) · [Facturación](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/billing).

## 1. Requisitos

Capacidad Fabric compatible, AI Functions habilitadas por la administración y sesión PySpark con SynapseML. Para Runtime 2.0 (Spark 4.1), revisar la configuración temporal indicada en Microsoft Learn. Adjuntar un Lakehouse solo si se desea persistir. Usar datos ficticios: el texto se procesa mediante el endpoint del modelo. Cada llamada puede consumir capacidad.

## 2. Configuración del modelo

In [6]:
from synapse.ml.spark import aifunc
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timezone
import uuid

MODEL_NAME = "gpt-5-mini"
aifunc.default_conf.set_deployment_name(MODEL_NAME)
print(aifunc.default_conf.get_deployment_name())

StatementMeta(, e0d26e48-dfad-4071-8b14-ec60a9b22124, 8, Finished, Available, Finished, False)

gpt-5-mini


## 2.1 Diagnóstico y parámetros
No se invoca el modelo en esta sección. No se presupone soporte de parámetros de razonamiento.

In [5]:
import inspect
from importlib.metadata import version
from pyspark import StorageLevel
print("Spark:", spark.version)
print("SynapseML:", version("synapseml"))
RUN_DEFAULT = True
RUN_CUSTOM = True
RUN_OPTIMIZED = True
RUN_PRODUCTIVE = True
MIN_TOKENS = 100
MAX_WORDS = 12
RATE_INPUT, RATE_CACHE, RATE_OUTPUT = 8.40, 0.84, 67.23
PERSIST_AUDIT = False
TABLE_NAME = "dbo.audit_ai_functions"
DETAIL_TABLE = "dbo.audit_ai_summarize_metrics"


StatementMeta(, e0d26e48-dfad-4071-8b14-ec60a9b22124, 7, Finished, Available, Finished, False)

Spark: 3.5.5.5.4.20260921.11
SynapseML: 1.1.3


## 2.2 Tokenizador local
Instalar tiktoken en el entorno Fabric si no está disponible. La instalación puede requerir reiniciar la sesión. La tokenización visible es una estimación y no incluye instrucciones ni razonamiento.

In [4]:
# Ejecutar solo si falta la dependencia y reiniciar si corresponde:
!pip install tiktoken
import tiktoken
try:
    encoding = tiktoken.encoding_for_model(MODEL_NAME)
except KeyError:
    encoding = tiktoken.get_encoding("o200k_base")
print("Tokenizador:", encoding.name)
def count_tokens(text):
    return None if text is None else len(encoding.encode(text, disallowed_special=()))


StatementMeta(, e0d26e48-dfad-4071-8b14-ec60a9b22124, 6, Finished, Available, Finished, False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 9.8 MB/s eta 0:00:00ta 0:00:01
Tokenizador: o200k_base


## 3. Dataset ficticio
Cada fila representa un incidente o solicitud. Los datos no corresponden a clientes reales. Se incluyen los campos de contexto para demostrar la diferencia entre resumir una columna y resumir toda la fila.

In [7]:
data = [
    (1, "Incidente", "Alta", "Resuelto", "Durante la carga nocturna, el pipeline de ventas falló por una conexión expirada. El equipo renovó las credenciales, reejecutó el proceso y verificó que los registros llegaron completos al destino."),
    (2, "Rendimiento", "Media", "En análisis", "El reporte comercial demora más de lo habitual. Se detectaron consultas con filtros poco selectivos. El equipo revisará el modelo semántico y realizará pruebas antes de aplicar cambios."),
    (3, "Solicitud", "Baja", "Pendiente", "El área financiera solicita un tablero que consolide ingresos y gastos mensuales. Se requiere definir los indicadores y validar las fuentes con los responsables del negocio."),
    (4, "Calidad", "Alta", "Mitigado", "Se identificaron registros duplicados en una tabla de pedidos. Se aisló la carga afectada y se aplicó una deduplicación temporal. Falta corregir la lógica de origen."),
    (5, "Disponibilidad", "Media", "Resuelto", "Una actualización del servicio dejó temporalmente indisponible el acceso a un conjunto de datos. El servicio se restauró y las pruebas confirmaron el acceso normal."),
    (6, "Soporte", "Baja", "Pendiente", None)
]
df = spark.createDataFrame(data, ["id", "tipo", "prioridad", "estado", "descripcion"])
display(df)

StatementMeta(, e0d26e48-dfad-4071-8b14-ec60a9b22124, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bb2b3f92-a46c-4c1c-a055-36ddb4286ae4)

In [8]:
print("Firma summarize:", inspect.signature(df.ai.summarize))
print("Configuración disponible:", [x for x in dir(aifunc.default_conf) if x.startswith(("set_", "get_"))])
assert {"input_col", "output_col", "error_col", "instructions"}.issubset(inspect.signature(df.ai.summarize).parameters), "Revisar firma del runtime"
# Dataset pequeño de laboratorio: tokenización en driver, sin UDF ni llamadas LLM.
source_rows = df.collect()
source_tokens = {r.id: count_tokens(r.descripcion) for r in source_rows}


StatementMeta(, e0d26e48-dfad-4071-8b14-ec60a9b22124, 10, Finished, Available, Finished, False)

Firma summarize: (*, input_col: str = None, output_col: str = None, error_col: str = None, input_col_type: str = 'text', col_types: dict = None, instructions: Optional[str] = None, **options) -> pyspark.sql.dataframe.DataFrame
Configuración disponible: ['get_URL', 'get_api_type', 'get_api_version', 'get_deployment_name', 'get_embedding_deployment_name', 'get_model', 'get_reasoning_effort', 'get_seed', 'get_subscription_key', 'get_temperature', 'get_top_p', 'get_verbosity', 'set_URL', 'set_api_type', 'set_api_version', 'set_deployment_name', 'set_embedding_deployment_name', 'set_model', 'set_reasoning_effort', 'set_seed', 'set_subscription_key', 'set_temperature', 'set_top_p', 'set_verbosity']


## 4. Funciones de ejecución y auditoría
Se materializan todas las columnas mediante collect (solo para este dataset pequeño). Se captura ai.stats una vez y se reconstruye un DataFrame local para las visualizaciones posteriores. Esto evita volver a evaluar el linaje de IA en las comparaciones. Para producción masiva, materializar en Delta en vez de collect. Las métricas incompletas se conservan como nulas; no se inventa consumo.

In [9]:
def read_stats(result):
    stats_df = result.ai.stats
    rows = stats_df.collect()
    if len(rows) != 1:
        raise RuntimeError("Revisar granularidad de ai.stats")
    display(spark.createDataFrame(rows, stats_df.schema))
    return rows[0].asDict(recursive=True)

def estimate_cu(stats):
    i, c, o = [stats.get(k) for k in ("input_tokens", "cached_tokens", "output_tokens")]
    if any(v is None for v in (i,c,o)) or not 0 <= c <= i or o < 0:
        return None
    return ((i-c)*RATE_INPUT + c*RATE_CACHE + o*RATE_OUTPUT)/1000

runs = {}
def execute_summary(mode, source, output, error, **kwargs):
    run_id = str(uuid.uuid4())
    timestamp = datetime.now(timezone.utc).isoformat()
    live = source.ai.summarize(output_col=output, error_col=error, **kwargs)
    live.persist(StorageLevel.MEMORY_AND_DISK)
    try:
        rows = live.collect()
        stats = read_stats(live)
        frozen = spark.createDataFrame(rows, live.schema)
    finally:
        live.unpersist()
    runs[mode] = dict(run_id=run_id, timestamp=timestamp, stats=stats,
                      rows=rows, result=frozen, output=output, error=error)
    print(mode, "CU-s estimados:", estimate_cu(stats))
    display(frozen)
    return frozen


StatementMeta(, e0d26e48-dfad-4071-8b14-ec60a9b22124, 11, Finished, Available, Finished, False)

## 5. Experimento estándar — línea base

In [10]:
if RUN_DEFAULT:
    resultado = execute_summary("DEFAULT", df, "resumen", "error_resumen", input_col="descripcion")
    stats_default = runs["DEFAULT"]["stats"]
    run_id_default = runs["DEFAULT"]["run_id"]


StatementMeta(, e0d26e48-dfad-4071-8b14-ec60a9b22124, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 73a6ddc0-7b52-42d7-95da-374bcfefd5f9)

DEFAULT CU-s estimados: 50.69709


SynapseWidget(Synapse.DataFrame, ed33c6bc-ea4a-419f-bf1a-a16bbcaded54)

## 6. Experimento ejecutivo — contexto multicolumna
Se conserva el experimento original para comparar; incluye ID y contexto, por lo que no aísla el efecto de las instrucciones.

In [11]:
if RUN_CUSTOM:
    df_ejecutivo = df.filter(F.col("descripcion").isNotNull()).select("id", "tipo", "prioridad", "estado", "descripcion")
    resultado_custom = execute_summary("CUSTOM", df_ejecutivo, "resumen_ejecutivo", "error_ejecutivo", instructions="Redacta en español un resumen ejecutivo de máximo 40 palabras. Incluye prioridad, estado, impacto y acciones pendientes si constan en los datos. No incluyas el identificador ni inventes información.")
    stats_custom = runs["CUSTOM"]["stats"]
    run_id_custom = runs["CUSTOM"]["run_id"]


StatementMeta(, e0d26e48-dfad-4071-8b14-ec60a9b22124, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7d383601-d6c9-4afe-9f4a-bb19f4359312)

CUSTOM CU-s estimados: 136.51464


SynapseWidget(Synapse.DataFrame, ac35e8c4-9749-4299-ab9a-cbfe0eb5c79c)

## 7. Experimento optimizado
Una sola columna, sin viñetas, máximo 12 palabras como objetivo (no garantía).

In [12]:
if RUN_OPTIMIZED:
    resultado_optimized = execute_summary("OPTIMIZED", df, "resumen_optimizado", "error_optimizado", input_col="descripcion", instructions=f"Resume en español en máximo {MAX_WORDS} palabras, una frase sin viñetas. Conserva problema y resolución o estado. No inventes datos.")
    stats_optimized = runs["OPTIMIZED"]["stats"]
    run_id_optimized = runs["OPTIMIZED"]["run_id"]


StatementMeta(, e0d26e48-dfad-4071-8b14-ec60a9b22124, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 428d458a-fa39-40c2-995f-596adc7584ce)

OPTIMIZED CU-s estimados: 105.79140000000001


SynapseWidget(Synapse.DataFrame, ff09c954-c973-42a8-8488-9f4bd740a297)

## 8. Estrategia productiva — evitar inferencias innecesarias
Umbral configurable de 100 tokens. Con los datos originales se espera que no se invoque IA. Si el resumen falla o no comprime, se conserva el original; ello no recupera los tokens ya consumidos.

In [13]:
if RUN_PRODUCTIVE:
    token_df = spark.createDataFrame([(k,v) for k,v in source_tokens.items()], "id long, tokens_original long")
    df_tokenizado = df.join(token_df, "id")
    eligible = df_tokenizado.filter(F.col("tokens_original") > MIN_TOKENS)
    eligible_ids = {r.id for r in eligible.select("id").collect()}
    generated = {}
    if eligible_ids:
        execute_summary("PRODUCTIVE", eligible, "resumen_ia", "error_ia", input_col="descripcion", instructions="Resume en español en una frase, reduce al menos 50%, conserva hechos y estado.")
        generated = {r.id:r for r in runs["PRODUCTIVE"]["rows"]}
    final_rows = []
    for r in source_rows:
        g = generated.get(r.id)
        text = g.resumen_ia if g is not None else None
        ok = g is not None and g.error_ia is None and bool(text) and count_tokens(text) < source_tokens[r.id]
        method = "AI_SUMMARIZE" if ok else "FALLBACK_ORIGINAL" if g is not None else "SIN_TEXTO" if r.descripcion is None else "ORIGINAL"
        final_rows.append((r.id,r.tipo,r.prioridad,r.estado,r.descripcion,source_tokens[r.id],text if ok else r.descripcion,method))
    df_final = spark.createDataFrame(final_rows, "id long, tipo string, prioridad string, estado string, descripcion string, tokens_original long, resumen string, metodo_resumen string")
    display(df_final)
    print("Filas elegibles:",len(eligible_ids),"Filas no enviadas:",len(source_rows)-len(eligible_ids),"CU-s LLM:",estimate_cu(runs["PRODUCTIVE"]["stats"]) if eligible_ids else 0)


StatementMeta(, e0d26e48-dfad-4071-8b14-ec60a9b22124, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0f5bb1b6-dddd-42fd-85ed-76e9c5cff1ca)

Filas elegibles: 0 Filas no enviadas: 6 CU-s LLM: 0


## 9. Compresión por registro
Comparar únicamente pares con texto original y respuesta válida. Una reducción negativa indica expansión. La fidelidad requiere revisión humana.

In [14]:
details = []
for mode, run in runs.items():
    for r in run["rows"]:
        original, summary = r.descripcion, r[run["output"]]
        a,b = count_tokens(original),count_tokens(summary)
        valid = original is not None and bool(summary) and r[run["error"]] is None
        details.append((run["run_id"],mode,r.id,original,summary,a,b,len(original) if original is not None else None,len(summary) if summary is not None else None,round(100*(1-b/a),2) if valid and a else None,valid,str(r[run["error"]]) if r[run["error"]] is not None else None))
detail_df = spark.createDataFrame(details,"run_id string, modalidad string, id long, descripcion string, resumen string, tokens_original long, tokens_resumen long, caracteres_original long, caracteres_resumen long, reduccion_pct double, valido boolean, error string")
display(detail_df.orderBy("id","modalidad"))
display(detail_df.filter(F.col("reduccion_pct") < 0))


StatementMeta(, e0d26e48-dfad-4071-8b14-ec60a9b22124, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, be053309-3692-486a-b3b8-5aee1823a1a9)

SynapseWidget(Synapse.DataFrame, 1a3d053c-3238-4a37-81df-03e530ce778a)

## 10. Auditoría de tokens y CU
Se conserva el esquema de auditoría original. El razonamiento forma parte de output_tokens; no se suma dos veces. El coste es estimado con las tarifas del notebook original, sin Spark. Se agrega una tabla de métricas por modalidad sin textos.

In [ ]:
audit_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("execution_timestamp_utc", StringType(), False),
    StructField("notebook_name", StringType(), False),
    StructField("operation", StringType(), False),
    StructField("model_configured", StringType(), True),
    StructField("model_reported", StringType(), True),
    StructField("num_input_rows", LongType(), True),
    StructField("num_successful", LongType(), True),
    StructField("num_exceptions", LongType(), True),
    StructField("num_unevaluated", LongType(), True),
    StructField("num_harmful", LongType(), True),
    StructField("input_tokens", LongType(), True),
    StructField("cached_tokens", LongType(), True),
    StructField("output_tokens", LongType(), True),
    StructField("reasoning_tokens", LongType(), True),
    StructField("estimated_cu_seconds", DoubleType(), True),
    StructField("client_type", StringType(), True),
    StructField("execution_status", StringType(), True)
])


audit_rows = []
metric_rows = []
for mode, run in runs.items():
    stats = run["stats"]
    def metric(k):
        return int(stats[k]) if stats.get(k) is not None else None
    total = len(run["rows"])
    invalid = sum(r[run["error"]] is not None or (r.descripcion is not None and not r[run["output"]]) for r in run["rows"])
    cu = estimate_cu(stats)
    audit_rows.append((run["run_id"],run["timestamp"],"NB_04_AI_SUMMARIZE","ai.summarize",MODEL_NAME,str(stats["model"]) if stats.get("model") is not None else None,total,metric("num_successful"),metric("num_exceptions"),metric("num_unevaluated"),metric("num_harmful"),metric("input_tokens"),metric("cached_tokens"),metric("output_tokens"),metric("reasoning_tokens"),cu,str(stats["client_type"]) if stats.get("client_type") is not None else None,"SUCCESS" if invalid==0 else "PARTIAL" if invalid<total else "INCOMPLETE"))
    pairs = [d for d in details if d[0]==run["run_id"] and d[10]]
    original = sum(d[5] for d in pairs)
    visible = sum(d[6] for d in pairs)
    all_original = sum(count_tokens(r.descripcion) or 0 for r in run["rows"])
    i,cache,o,reason = [metric(k) for k in ("input_tokens","cached_tokens","output_tokens","reasoning_tokens")]
    metric_rows.append((run["run_id"],mode,len(pairs),all_original,original,visible,i,cache,o,reason,i+o if i is not None and o is not None else None,i-all_original if i is not None else None,o-reason if o is not None and reason is not None and 0<=reason<=o else None,round(100*(1-visible/original),2) if original else None,round((i+o)/all_original,2) if all_original and i is not None and o is not None else None,round(100*reason/o,2) if o and reason is not None and 0<=reason<=o else None,(i-cache)*RATE_INPUT/1000 if cu is not None else None,cache*RATE_CACHE/1000 if cu is not None else None,o*RATE_OUTPUT/1000 if cu is not None else None,cu))
audit_df = spark.createDataFrame(audit_rows,audit_schema)
metrics_df = spark.createDataFrame(metric_rows,"run_id string, modalidad string, pares_validos long, tokens_original_en_filas long, tokens_original_pares long, tokens_resumen_pares long, input_tokens long, cached_tokens long, output_tokens long, reasoning_tokens long, total_tokens long, overhead_entrada_estimado long, output_sin_razonamiento long, reduccion_pct double, factor_consumo double, reasoning_pct double, cu_input double, cu_cache double, cu_output double, estimated_cu_seconds double")
display(audit_df)
display(metrics_df)
display(audit_df.agg(F.sum("estimated_cu_seconds").alias("cu_s_conocidos"),F.count(F.when(F.col("estimated_cu_seconds").isNull(),1)).alias("ejecuciones_sin_estimacion")))


## 11. Persistencia opcional en Delta
Activar solo con Lakehouse adjunto. MERGE por run_id evita duplicar la misma captura al repetir esta celda; nuevas inferencias generan nuevos identificadores. No se guardan descripciones ni respuestas en estas tablas.

In [ ]:
if PERSIST_AUDIT:
    from delta.tables import DeltaTable
    for name, frame in [(TABLE_NAME,audit_df),(DETAIL_TABLE,metrics_df)]:
        if not spark.catalog.tableExists(name):
            frame.write.format("delta").mode("errorifexists").saveAsTable(name)
        else:
            DeltaTable.forName(spark,name).alias("t").merge(frame.alias("s"),"t.run_id = s.run_id").whenNotMatchedInsertAll().execute()


## 11. Extensión multimodal (opcional)
Microsoft documenta el uso de `input_col_type="path"` para resumir archivos compatibles, como PDF o imágenes, mediante una columna con rutas. Revisar requisitos y disponibilidad en la [documentación multimodal](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/multimodal-overview). El siguiente bloque no ejecuta inferencias por defecto; si se habilita, su consumo debe auditarse por separado.

In [ ]:
RUN_MULTIMODAL = False
if RUN_MULTIMODAL:
    file_path = "Files/ejemplo.pdf"  # Ajustar a un archivo compatible y accesible.
    archivos = spark.createDataFrame([(file_path,)], ["ruta"])
    resumen_archivo = archivos.ai.summarize(input_col="ruta", input_col_type="path", output_col="resumen", error_col="error", instructions="Resume en español los hechos principales del documento.")
    resumen_archivo.cache()
    resumen_archivo.collect()
    display(resumen_archivo)
    stats_archivo = read_stats(resumen_archivo)
    print("CU-segundos estimados (no incluidos en auditoría principal):", estimate_cu(stats_archivo))
    resumen_archivo.unpersist()

## 13. Conclusiones
Comparar compresión visible, tokens totales y CU estimados. El filtrado previo evita inferencias; reducir palabras no garantiza reducir razonamiento. El overhead incluye serialización y contexto, no solo instrucciones. Revisar Capacity Metrics y fidelidad semántica. Las tres modalidades experimentales generan consumo adicional; desactivarlas para usar solo la estrategia productiva.

**Siguiente notebook:** NB_05 — AI_TRANSLATE.